## 1. Tiện ích chung (metrics, DM, walk-forward harness)

In [15]:
# ============================================================
# 1) TIEN ICH CHUNG
# ============================================================
import os
import numpy as np
import pandas as pd
from math import erf, sqrt
import warnings, time
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, r"E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\scripts\\model")  # noi de export_preds.py
from export_preds import save_preds
PREDS_DIR = r"E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\preds"

# ---------- logging / progress ----------
def _now():
    return time.strftime("%H:%M:%S")

try:
    from tqdm import tqdm                 # pip install tqdm
    def progress(it, desc=""):
        return tqdm(list(it), desc=desc, leave=False)
except Exception:                          # fallback neu khong co tqdm
    def progress(it, desc=""):
        items = list(it); total = len(items)
        for i, x in enumerate(items, 1):
            if i == 1 or i % 20 == 0 or i == total:
                print(f"    [{_now()}] {desc}: {i}/{total}", flush=True)
            yield x

DATA_PATH    = "E:\FPT\AI\SEM8_AI\DAP391m\project\CofPred\data\processed\gia_cafe_master_full.csv"
TARGET       = "Gia_target"
DATE_COL     = "date"          # tu dong nhan dien neu ten khac (vd "Ngay")
HORIZONS     = [1, 5, 21, 63]
INITIAL_FRAC = 0.6
STEP         = 5               # tang len de chay nhanh hon

# 24 feature dung trong benchmark CU (ML). Giu de can chinh tap danh gia cho KHOP.
FEATURES_ALL = [
    "target_lag1", "target_lag2", "target_lag3", "target_ret_lag1", "MA5", "MA10", "std5", "dayofweek", "month",
    "london_vnd_kg_lag1", "usdvnd_lag1", "diesel", "diesel_chg_1m", "diesel_chg_3m", "Luong_lag1m", "rain_90d", "oni", "waterbal_90d",
    "area_tn", "prod_tn", "yield_tn", "tonkho_tan", "dongia_lag1m", "dongia_ret_lag1m",
]

def load_data():
    global DATE_COL
    df = pd.read_csv(DATA_PATH)
    if DATE_COL not in df.columns:                      # tu dong nhan cot ngay
        for c in ["date", "Date", "Ngay", "ngay"]:
            if c in df.columns:
                DATE_COL = c; break
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])
        df = df.sort_values(DATE_COL).reset_index(drop=True)
    # === CAN CHINH VOI BENCHMARK CU ===
    # Code cu danh gia tren ma tran dac trung da dropna 24 feature (n=1516, start=909).
    # Phai dung CUNG tap hang nay thi MAE Naive moi khop (1567 @h=1, 8646 @h=21).
    feats = [c for c in FEATURES_ALL if c in df.columns]
    df["prev_price"] = df[TARGET].shift(1)
    before = len(df)
    df = df.dropna(subset=[TARGET] + feats + ["prev_price"]).reset_index(drop=True)
    print(f"[load] giu {len(df)}/{before} hang sau dropna {len(feats)} feature "
          f"(start={int(len(df) * INITIAL_FRAC)}) -> khop benchmark cu", flush=True)
    return df

# ---------- metrics ----------
def mae(y, yhat):  return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(yhat, float)) ** 2)))

def directional_accuracy(base, y_true, y_pred):
    dt = np.sign(np.asarray(y_true, float) - np.asarray(base, float))
    dp = np.sign(np.asarray(y_pred, float) - np.asarray(base, float))
    return float(np.mean(dt == dp))

# ---------- Diebold-Mariano (khong can scipy) ----------
def diebold_mariano(e_model, e_naive, h=1, loss="mae"):
    e1, e2 = np.asarray(e_model, float), np.asarray(e_naive, float)
    d = (np.abs(e1) - np.abs(e2)) if loss == "mae" else (e1 ** 2 - e2 ** 2)
    n = len(d)
    if n == 0:
        return float("nan"), float("nan")
    dbar = d.mean()
    var = np.var(d, ddof=0)
    for k in range(1, h):
        if n - k > 1:
            cov = np.cov(d[k:], d[:-k])[0, 1]
            var += 2 * (1 - k / h) * cov
    if var <= 0:
        return float("nan"), float("nan")
    dm = dbar / sqrt(var / n)
    p = 2 * (1 - 0.5 * (1 + erf(abs(dm) / sqrt(2))))
    return float(dm), float(p)

# ---------- walk-forward harness (mot bien) ----------
def walk_forward_eval(serie, forecast_fn, horizon, exog=None, desc="walk-forward"):
    # Quy uoc GIONG benchmark cu: tai ngay quyet dinh d, biet du lieu [0..d],
    # du bao gia(d+h). Naive = gia(d). Exog giu co dinh = gia tri ngay d (khong look-ahead).
    n = len(serie)
    start = int(n * INITIAL_FRAC)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(sorted(range(n - 1 - horizon, start - 1, -STEP)), desc):
        train_y = serie[:d + 1]
        tgt = d + horizon
        tr_ex = exog[:d + 1] if exog is not None else None
        fu_ex = np.tile(exog[d:d + 1], (horizon, 1)) if exog is not None else None
        try:
            yhat = forecast_fn(train_y, tr_ex, fu_ex, horizon)
        except Exception:
            yhat = train_y[-1]          # fallback = naive
        y_pred.append(float(yhat))
        y_true.append(float(serie[tgt]))
        y_naive.append(float(serie[d]))
        base.append(float(serie[d]))
    return {k: np.asarray(v, float) for k, v in
            dict(y_true=y_true, y_pred=y_pred, y_naive=y_naive, base=base).items()}

def summarize(tag, horizon, r):
    e_m = r["y_true"] - r["y_pred"]
    e_n = r["y_true"] - r["y_naive"]
    dm, p = diebold_mariano(e_m, e_n, h=horizon, loss="mae")
    n_mae = mae(r["y_true"], r["y_naive"])
    m_mae = mae(r["y_true"], r["y_pred"])
    return dict(model=tag, h=horizon, n=len(r["y_true"]),
                MAE=round(m_mae, 1), MAE_naive=round(n_mae, 1),
                dMAE_pct=round(100 * (m_mae / n_mae - 1), 2),
                RMSE=round(rmse(r["y_true"], r["y_pred"]), 1),
                DA=round(directional_accuracy(r["base"], r["y_true"], r["y_pred"]), 3),
                DM=round(dm, 3), DM_p=round(p, 4))

## 2. Hàm dự báo từng model (Naive, AutoARIMA, SARIMA, SARIMAX)

In [16]:
# ============================================================
# 2) HAM DU BAO (mot bien + exog)
# ============================================================
import pmdarima as pm
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fc_naive(train_y, tr_ex, fu_ex, h):
    return train_y[-1]

def fc_autoarima(train_y, tr_ex, fu_ex, h):
    m = pm.auto_arima(train_y, seasonal=False, d=None,
                      max_p=5, max_q=5, suppress_warnings=True,
                      error_action="ignore", stepwise=True)
    return float(np.asarray(m.predict(n_periods=h))[-1])

def fc_sarima(train_y, tr_ex, fu_ex, h):
    m = pm.auto_arima(train_y, seasonal=True, m=5, d=None, D=None,
                      max_p=3, max_q=3, max_P=2, max_Q=2,
                      suppress_warnings=True, error_action="ignore", stepwise=True)
    return float(np.asarray(m.predict(n_periods=h))[-1])

def make_fc_sarimax(order=(1, 1, 1), seasonal=(0, 0, 0, 0)):
    def _fc(train_y, tr_ex, fu_ex, h):
        res = SARIMAX(train_y, exog=tr_ex, order=order, seasonal_order=seasonal,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        return float(np.asarray(res.forecast(steps=h, exog=fu_ex))[-1])
    return _fc

## 3. VECM (đa biến: nội địa ↔ London) 

In [17]:
# ============================================================
# 3) VECM
# ============================================================
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank

def walk_forward_vecm(df, cols=("Gia_target", "london_vnd_kg"), horizon=1, k_ar_diff=1):
    data = df[list(cols)].dropna().astype(float).reset_index(drop=True).values
    serie = data[:, 0]
    n = len(data)
    start = int(n * INITIAL_FRAC)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(sorted(range(n - 1 - horizon, start - 1, -STEP)), f"VECM h={horizon}"):
        train = data[:d + 1]
        try:
            r = max(select_coint_rank(train, det_order=0, k_ar_diff=k_ar_diff, signif=0.05).rank, 1)
            res = VECM(train, k_ar_diff=k_ar_diff, coint_rank=r, deterministic="ci").fit()
            f = float(res.predict(steps=horizon)[:, 0][-1])
        except Exception:
            f = float(serie[d])
        y_pred.append(f); y_true.append(float(serie[d + horizon]))
        y_naive.append(float(serie[d])); base.append(float(serie[d]))
    return {k: np.asarray(v, float) for k, v in
            dict(y_true=y_true, y_pred=y_pred, y_naive=y_naive, base=base).items()}

## 4. ARIMA-GARCH — đánh giá độ phủ (coverage) khoảng 95%

In [18]:
# ============================================================
# 4) ARIMA-GARCH: coverage khoang tin cay
# ============================================================
from arch import arch_model

def garch_coverage(serie, horizon=1):
    ret = pd.Series(serie).pct_change().dropna().values * 100.0
    n = len(ret); start = int(n * INITIAL_FRAC)
    inside = total = 0
    for d in progress(range(start, n - horizon + 1, STEP), f"GARCH h={horizon}"):
        try:
            res = arch_model(ret[:d], mean="AR", lags=1, vol="GARCH",
                             p=1, q=1, dist="t").fit(disp="off")
            fc = res.forecast(horizon=horizon, reindex=False)
            mu = fc.mean.values[-1]
            sd = np.sqrt(fc.variance.values[-1])
            cum_mu = mu.sum()
            cum_sd = np.sqrt((sd ** 2).sum())          # gia su loi suat doc lap
            lo, hi = cum_mu - 1.96 * cum_sd, cum_mu + 1.96 * cum_sd
            actual = (serie[d + horizon - 1] / serie[d - 1] - 1.0) * 100.0
            inside += int(lo <= actual <= hi); total += 1
        except Exception:
            continue
    return inside / total if total else float("nan")

## 5. Deep Learning — NHITS / NBEATSx (neuralforecast)

In [19]:
# ============================================================
# 5) DL: NHITS / NBEATSx (cross_validation walk-forward san co)
# ============================================================
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, NBEATSx
from neuralforecast.losses.numpy import mae as nf_mae

def run_dl(df, horizon=63, exog=("london_vnd_kg_lag1", "usdvnd_lag1")):
    long = pd.DataFrame({
        "unique_id": "cafe",
        "ds": pd.to_datetime(df[DATE_COL]) if DATE_COL in df else pd.RangeIndex(len(df)),
        "y": df[TARGET].astype(float).values,
    })
    exog = [c for c in exog if c in df.columns]
    for c in exog:
        long[c] = df[c].astype(float).ffill().values
    models = [
        NHITS(h=horizon, input_size=2 * horizon, max_steps=500,
              futr_exog_list=exog, scaler_type="robust"),
        NBEATSx(h=horizon, input_size=2 * horizon, max_steps=500,
                futr_exog_list=exog, scaler_type="robust"),
    ]
    nf = NeuralForecast(models=models, freq="B")
    cv = nf.cross_validation(df=long, n_windows=30, step_size=STEP)
    for col in ["NHITS", "NBEATSx"]:
        print(col, "MAE =", round(nf_mae(cv["y"].values, cv[col].values), 1))
    return cv

## 6. Foundation model zero-shot — Chronos (không train)

In [20]:
# ============================================================
# 6) Chronos zero-shot
# ============================================================
import torch
from chronos import ChronosPipeline

def run_chronos(serie, horizon=63, context=512):
    pipe = ChronosPipeline.from_pretrained(
        "amazon/chronos-bolt-base", device_map="cpu", torch_dtype=torch.float32)
    n = len(serie); start = int(n * INITIAL_FRAC)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(sorted(range(n - 1 - horizon, start - 1, -STEP)), f"Chronos h={horizon}"):
        ctx = torch.tensor(serie[max(0, d - context):d + 1], dtype=torch.float32)
        fc = pipe.predict(ctx, prediction_length=horizon)
        med = np.quantile(fc[0].numpy(), 0.5, axis=0)[-1]
        y_pred.append(float(med)); y_true.append(float(serie[d + horizon]))
        y_naive.append(float(serie[d])); base.append(float(serie[d]))
    return {k: np.asarray(v, float) for k, v in
            dict(y_true=y_true, y_pred=y_pred, y_naive=y_naive, base=base).items()}

## 7. Chạy tất cả & xuất bảng so sánh

In [21]:
from export_preds import save_preds
# ============================================================
# 7) RUNNER
# ============================================================
def run_all():
    df = load_data()                                # da loc feature-complete (n=1516)
    serie = df[TARGET].astype(float).values

    exog_cols = [c for c in ["london_vnd_kg_lag1", "usdvnd_lag1", "oni", "rain_90d"] if c in df.columns]
    exog = df[exog_cols].astype(float).ffill().values if exog_cols else None
    fc_sarimax = make_fc_sarimax(order=(1, 1, 1))

    # tu dong tim cot gia London cho VECM (ten cot trong file co the khac)
    london = next((c for c in ["london_vnd_kg", "london_vnd_kg_lag1"] if c in df.columns),
                  next((c for c in df.columns if "london" in c.lower()), None))

    # cong tac bat/tat tung model (SARIMA m=5 rat cham -> mac dinh TAT)
    USE = {"Naive": True, "AutoARIMA": True, "SARIMA_m5": False,
           "SARIMAX": True, "VECM": True}

    # danh sach job DOC LAP: (nhan, horizon, ham_tra_ve_dict_ket_qua)
    jobs = []
    for h in HORIZONS:
        if USE["Naive"]:
            jobs.append((f"Naive h={h}",     h, lambda h=h: walk_forward_eval(serie, fc_naive,     h, desc=f"Naive h={h}")))
        if USE["AutoARIMA"]:
            jobs.append((f"AutoARIMA h={h}", h, lambda h=h: walk_forward_eval(serie, fc_autoarima, h, desc=f"AutoARIMA h={h}")))
        if USE["SARIMA_m5"]:
            jobs.append((f"SARIMA_m5 h={h}", h, lambda h=h: walk_forward_eval(serie, fc_sarima,    h, desc=f"SARIMA_m5 h={h}")))
        if USE["SARIMAX"] and exog is not None:
            jobs.append((f"SARIMAX h={h}",   h, lambda h=h: walk_forward_eval(serie, fc_sarimax,   h, exog=exog, desc=f"SARIMAX h={h}")))
        if USE["VECM"] and london is not None:
            jobs.append((f"VECM h={h}",      h, lambda h=h: walk_forward_vecm(df, cols=("Gia_target", london), horizon=h)))

    if USE["VECM"] and london is None:
        print("[!] Khong tim thay cot gia London -> BO QUA VECM. Cot hien co:", list(df.columns), flush=True)
    os.makedirs("E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\TS_model\\", exist_ok=True)
    out_path = "E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\TS_model\\ts_model_comparison.csv"
    total = len(jobs); done = 0; t0 = time.time(); rows = []
    for tag, h, fn in jobs:
        done += 1; t1 = time.time()
        print(f"[{_now()}] ({done}/{total}) {tag:16s} dang chay...", flush=True)
        try:
            r = fn()                                   # dict: y_true, y_pred, y_naive, base
            row = summarize(tag.split(" h=")[0], h, r)
            # --- xuat preds tung diem cho MCS ---
            n = len(serie); start = int(n * INITIAL_FRAC)
            dates_all = pd.to_datetime(df[DATE_COL]).values
            origins = sorted(range(n - 1 - h, start - 1, -STEP))  # trung khop harness (d chay tren range nay)
            dts = dates_all[[o + h for o in origins]]   # ngay cua diem duoc du bao (d+h)
            save_preds(tag.split(" h=")[0], h, dts, r['y_true'], r['y_pred'], outdir=PREDS_DIR)
            rows.append(row)
            pd.DataFrame(rows).to_csv(out_path, index=False)  # ghi NGAY sau moi model -> luon co ket qua
            print(f"           OK  {time.time()-t1:7.1f}s | MAE={row['MAE']}  dMAE%={row['dMAE_pct']}"
                  f"  RMSE={row['RMSE']}  DA={row['DA']}  DM_p={row['DM_p']}", flush=True)
        except Exception as e:
            print(f"           LOI {time.time()-t1:7.1f}s: {type(e).__name__}: {e}", flush=True)

    print(f"\n[{_now()}] HOAN TAT {len(rows)}/{total} job trong {time.time()-t0:.1f}s")
    if rows:
        out = pd.DataFrame(rows).sort_values(["h", "MAE"]).reset_index(drop=True)
        out.to_csv(out_path, index=False)
        print("\n=== BANG KET QUA ===")
        print(out.to_string(index=False))

    print(f"\n[{_now()}] GARCH coverage 95%:")
    for h in HORIZONS:
        try:
            print(f"  [{_now()}] h={h:>2}: {garch_coverage(serie, h):.3f}", flush=True)
        except Exception as e:
            print(f"  [{_now()}] h={h:>2}: LOI {type(e).__name__}: {e}", flush=True)
    return pd.DataFrame(rows)

if __name__ == "__main__":
    run_all()

[load] giu 1516/1566 hang sau dropna 24 feature (start=909) -> khop benchmark cu
[11:43:19] (1/16) Naive h=1        dang chay...


  [save_preds] E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\preds\Naive__h1.csv  (122 diem)
           OK      0.0s | MAE=1567.3  dMAE%=0.0  RMSE=3811.2  DA=0.098  DM_p=nan
[11:43:19] (2/16) AutoARIMA h=1    dang chay...


  [save_preds] E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\preds\AutoARIMA__h1.csv  (122 diem)
           OK    556.2s | MAE=1594.9  dMAE%=1.76  RMSE=3870.5  DA=0.377  DM_p=0.3909
[11:52:36] (3/16) SARIMAX h=1      dang chay...


  [save_preds] E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\preds\SARIMAX__h1.csv  (122 diem)
           OK     55.0s | MAE=1607.9  dMAE%=2.59  RMSE=3893.5  DA=0.451  DM_p=0.1497
[11:53:30] (4/16) VECM h=1         dang chay...


  [save_preds] E:\\FPT\\AI\\SEM8_AI\\DAP391m\\project\\CofPred\\results\\preds\VECM__h1.csv  (122 diem)
           OK      2.0s | MAE=1634.6  dMAE%=4.29  RMSE=3841.0  DA=0.467  DM_p=0.0976
[11:53:32] (5/16) Naive h=5        dang chay...


...


[12:25:14] HOAN TAT 16/16 job trong 2514.3s

=== BANG KET QUA ===
    model  h   n     MAE  MAE_naive  dMAE_pct    RMSE    DA    DM   DM_p
    Naive  1 122  1567.3     1567.3      0.00  3811.2 0.098   NaN    NaN
AutoARIMA  1 122  1594.9     1567.3      1.76  3870.5 0.377 0.858 0.3909
  SARIMAX  1 122  1607.9     1567.3      2.59  3893.5 0.451 1.441 0.1497
     VECM  1 122  1634.6     1567.3      4.29  3841.0 0.467 1.657 0.0976
    Naive  5 121  3508.1     3508.1      0.00  5587.9 0.008   NaN    NaN
  SARIMAX  5 121  3611.4     3508.1      2.94  5944.1 0.430 1.366 0.1720
AutoARIMA  5 121  3693.0     3508.1      5.27  6183.3 0.554 0.804 0.4213
     VECM  5 121  3704.6     3508.1      5.60  5956.9 0.537 1.461 0.1439
    Naive 21 118  8646.5     8646.5      0.00 11385.4 0.000   NaN    NaN
  SARIMAX 21 118  9401.3     8646.5      8.73 13245.7 0.466 1.104 0.2695
AutoARIMA 21 118  9633.6     8646.5     11.42 13458.3 0.466 1.203 0.2289
     VECM 21 118 10455.4     8646.5     20.92 13509.5 0.4

  [12:25:14] h= 1: 0.984


  [12:25:17] h= 5: 0.843


  [12:25:20] h=21: 0.712


  [12:25:23] h=63: 0.706
